In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Week 6 Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving ecommerce_orders.csv to ecommerce_orders.csv


In [4]:
df = spark.read.csv(
    "ecommerce_orders.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!\n")

df.show(5)

Dataset Loaded Successfully!

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


In [5]:
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [6]:
from pyspark.sql.functions import col

electronics_df = df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
)

electronics_df.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      P101| 178.18|
|      P102| 1752.3|
|      P103| 343.38|
|      P110| 227.74|
|      P111|1036.04|
|      P113|1589.46|
|      P122|1047.84|
|      P137|2305.72|
|      P142| 264.32|
+----------+-------+



In [7]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DoubleType

df_string = df.withColumn(
    "price",
    col("price").cast(StringType())
)

updated_df = df_string.withColumnRenamed(
    "old_name",
    "new_name"
).withColumn(
    "price",
    col("price").cast(DoubleType())
)

updated_df.printSchema()
updated_df.show(5)

root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|new_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  I

In [8]:
from pyspark.sql.functions import col

completed_orders = df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)
completed_orders.show()

+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price| amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485| 8761.5|Completed|  West|    High|
|   U015|      P115|    Grocery| Item_15|2154.68|      1826|2154.68|Completed| North|  Medium|
|   U017|      P117|  Furniture| Item_17| 1073.8|       910| 4295.2|Completed|  East|    High|
|   U020|      P120|   Clothing| Item_20|1348.74|      1143|5394.96|Completed| North|    High|
|   U029|      P129|    Grocery| Item_29| 1298.0|      1100| 1298.0|Completed|  East|  Medium|
|   U038|      P138|   Clothing| Item_38|  790.6|       670| 3162.4|Completed| South|     Low|
|   U039|      P139|     Sports| Item_39| 1510.4|      1280| 6041.6|Completed|  West|  Medium|
|   U045|      P145|  Furniture| Item_45|2069.72| 

In [9]:
from pyspark.sql.functions import col

final_price_df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)
final_price_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+------------------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|       final_price|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+------------------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|178.17999999999998|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|            1752.3|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|            343.38|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|1847.8799999999999|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|            789.42|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+---

In [10]:
df.write.mode("overwrite").parquet("input_parquet")

In [11]:
parquet_df = spark.read.parquet("input_parquet")

parquet_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


In [12]:
from pyspark.sql.functions import col

filtered_df = parquet_df.filter(
    col("user_id").isNotNull()
)

filtered_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


In [13]:
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output_csv")

In [14]:
import os

print(os.listdir("output_csv"))

['._SUCCESS.crc', 'part-00000-a2e00ee8-fc0e-4661-b2d0-d4c35fc5709e-c000.csv', '.part-00000-a2e00ee8-fc0e-4661-b2d0-d4c35fc5709e-c000.csv.crc', '_SUCCESS']


In [15]:
from pyspark.sql.functions import col

filtered_region_priority = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

filtered_region_priority.show()

+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price| amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151| 534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485| 8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291| 686.76|Completed| North|     Low|
|   U006|      P106|    Grocery|  Item_6| 493.24|       418| 986.48|  Pending| North|    High|
|   U007|      P107|  Furniture|  Item_7| 351.64|       298|1054.92|  Pending|  East|    High|
|   U008|      P108|  Furniture|  Item_8|1413.64|      1198|1413.64|  Pending| North|     Low|
|   NULL|      P110|Electronics| Item_10| 227.74|       193| 455.48|  Pending| North|    High|
|   U011|      P111|Electronics| Item_11|1036.04| 